In [1]:
%pip install scikit-learn pandas numpy optuna xgboost lightgbm


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

TRAIN_DATA = pd.read_csv('train-data.csv', index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA = pd.read_csv('test-data.csv', index_col='id')

In [3]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

if 'subscription' in TRAIN_DATA.columns:
    TRAIN_DATA = TRAIN_DATA.drop(columns=['subscription'])

def add_features(df):
    df = df.copy()
    # Previous campaign
    df['contacted_recently'] = ((df['pdays'] != -1) & (df['pdays'] < 30)).astype(int)
    df['prev_success'] = (df['poutcome'] == 'SUC').astype(int)
    df['never_contacted'] = (df['previous'] == 0).astype(int)
    df['pdays_recent'] = df['pdays'].apply(lambda x: 0 if x == -1 else x)
    df['multiple_prev_contacts'] = (df['previous'] > 2).astype(int)
    # Call duration
    df['very_short_call'] = (df['duration'] < 30).astype(int)
    df['short_call'] = (df['duration'] < 60).astype(int)
    df['medium_call'] = ((df['duration'] >= 60) & (df['duration'] <= 300)).astype(int)
    df['long_call'] = (df['duration'] > 300).astype(int)
    df['very_long_call'] = (df['duration'] > 600).astype(int)
    # Duration buckets as a single ordinal feature
    df['duration_bucket'] = pd.cut(
        df['duration'],
        bins=[-1, 30, 60, 180, 300, 600, 99999],
        labels=[0, 1, 2, 3, 4, 5]
    ).astype(int)
    # Balance
    df['debt'] = (df['balance'] < 0).astype(int)
    df['has_balance'] = (df['balance'] > 0).astype(int)
    df['medium_balance'] = ((df['balance'] > 0) & (df['balance'] <= 1000)).astype(int)
    df['high_balance'] = (df['balance'] > 1000).astype(int)
    df['very_high_balance'] = (df['balance'] > 5000).astype(int)
    df['log_balance'] = np.log1p(df['balance'].clip(lower=0))
    # Campaign pressure
    df['first_contact'] = (df['campaign'] == 1).astype(int)
    df['over_contacted'] = (df['campaign'] > 5).astype(int)
    df['log_campaign'] = np.log1p(df['campaign'])
    # Age
    df['is_young'] = (df['age'] < 30).astype(int)
    df['is_middle_age'] = ((df['age'] >= 30) & (df['age'] <= 60)).astype(int)
    df['is_retired_age'] = (df['age'] > 60).astype(int)
    # Interactions
    df['long_call_prev_success'] = df['long_call'] * df['prev_success']
    df['long_call_never_contacted'] = df['long_call'] * df['never_contacted']
    df['high_balance_long_call'] = df['high_balance'] * df['long_call']
    df['success_signal'] = ((df['duration'] > 300) & (df['poutcome'] == 'SUC')).astype(int)
    df['warm_lead'] = ((df['contacted_recently'] == 1) & (df['prev_success'] == 1)).astype(int)
    df['cold_lead'] = ((df['never_contacted'] == 1) & (df['short_call'] == 1)).astype(int)
    # Month/quarter
    df['q1'] = df['month'].isin([1, 2, 3]).astype(int)
    df['q2'] = df['month'].isin([4, 5, 6]).astype(int)
    df['q3'] = df['month'].isin([7, 8, 9]).astype(int)
    df['q4'] = df['month'].isin([10, 11, 12]).astype(int)
    return df

TRAIN_DATA = add_features(TRAIN_DATA)
TEST_DATA  = add_features(TEST_DATA)

print('TRAIN_DATA shape:', TRAIN_DATA.shape)
print('TEST_DATA shape: ', TEST_DATA.shape)

TRAIN_DATA shape: (29839, 49)
TEST_DATA shape:  (19893, 49)


In [4]:
from sklearn.preprocessing import OrdinalEncoder

cat_cols = ['job', 'marital_status', 'education', 'default_loan',
            'housing_loan', 'personal_loan', 'contact_type', 'poutcome']

num_cols = ['age', 'balance', 'day', 'month', 'duration',
            'campaign', 'pdays', 'previous',
            'contacted_recently', 'prev_success', 'never_contacted',
            'pdays_recent', 'multiple_prev_contacts',
            'very_short_call', 'short_call', 'medium_call',
            'long_call', 'very_long_call', 'duration_bucket',
            'debt', 'has_balance', 'medium_balance',
            'high_balance', 'very_high_balance', 'log_balance',
            'first_contact', 'over_contacted', 'log_campaign',
            'is_young', 'is_middle_age', 'is_retired_age',
            'long_call_prev_success', 'long_call_never_contacted',
            'high_balance_long_call', 'success_signal',
            'warm_lead', 'cold_lead',
            'q1', 'q2', 'q3', 'q4']

ENCODER = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
).fit(TRAIN_DATA[cat_cols])

def preprocess(df, encoder):
    df = df.copy()
    cat_enc = pd.DataFrame(
        encoder.transform(df[cat_cols]),
        columns=cat_cols,
        index=df.index
    )
    num_df = df[num_cols].copy()
    return pd.concat([cat_enc, num_df], axis=1)

X_train = preprocess(TRAIN_DATA, ENCODER)
X_test  = preprocess(TEST_DATA,  ENCODER)
y_train = TRAIN_LABEL['subscription'].values

print('X_train shape:', X_train.shape)
print('X_test  shape:', X_test.shape)
print('Class distribution — 0:', (y_train==0).sum(), '| 1:', (y_train==1).sum())

X_train shape: (29839, 49)
X_test  shape: (19893, 49)
Class distribution — 0: 26353 | 1: 3486


In [5]:
from sklearn.model_selection import StratifiedKFold

def target_encode_cv(X_tr, y_tr, X_te, cols, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    X_tr_enc = X_tr.copy()
    X_te_enc  = X_te.copy()
    global_mean = y_tr.mean()

    for col in cols:
        oof     = np.full(len(X_tr), global_mean)
        te_vals = np.zeros(len(X_te))

        for fold_tr_idx, fold_val_idx in skf.split(X_tr, y_tr):
            means = (
                y_tr.iloc[fold_tr_idx]
                .groupby(X_tr[col].iloc[fold_tr_idx])
                .mean()
            )
            oof[fold_val_idx] = (
                X_tr[col].iloc[fold_val_idx]
                .map(means).fillna(global_mean).values
            )
            te_vals += (
                X_te[col].reset_index(drop=True)
                .map(means).fillna(global_mean).values / n_splits
            )

        X_tr_enc[col + '_te'] = oof
        X_te_enc[col  + '_te'] = te_vals

    return X_tr_enc, X_te_enc

y_series = pd.Series(y_train, index=X_train.index)
X_train_te, X_test_te = target_encode_cv(X_train, y_series, X_test, cat_cols)

print('X_train_te shape:', X_train_te.shape)
print('X_test_te  shape:', X_test_te.shape)

X_train_te shape: (29839, 57)
X_test_te  shape: (19893, 57)


In [6]:
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import balanced_accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

X_arr    = X_train_te.values
X_te_arr = X_test_te.values

# ── Optuna HPO for XGBoost ────────────────────────────────────────────────────
def xgb_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 500, 2000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth':         trial.suggest_int('max_depth', 3, 8),
        'min_child_weight':  trial.suggest_int('min_child_weight', 5, 50),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 3.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.5, 5.0),
        'gamma':             trial.suggest_float('gamma', 0.0, 2.0),
        'scale_pos_weight':  scale_pos,
        'eval_metric':       'logloss',
        'random_state':      42,
        'n_jobs':            -1,
    }
    model = XGBClassifier(**params)
    skf_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(
        model, X_arr, y_train,
        cv=skf_inner, scoring='balanced_accuracy', n_jobs=-1
    )
    return scores.mean()

xgb_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
xgb_study.optimize(xgb_objective, n_trials=60, show_progress_bar=True)
print(f'Best XGB Optuna CV: {xgb_study.best_value:.4f}')
print(f'Best XGB params:    {xgb_study.best_params}')

# ── Optuna HPO for LightGBM ───────────────────────────────────────────────────
def lgbm_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 500, 2000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth':         trial.suggest_int('max_depth', 3, 8),
        'num_leaves':        trial.suggest_int('num_leaves', 15, 100),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 3.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.5, 5.0),
        'class_weight':      'balanced',
        'random_state':      42,
        'n_jobs':            -1,
        'verbose':           -1,
    }
    model = LGBMClassifier(**params)
    skf_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(
        model, X_arr, y_train,
        cv=skf_inner, scoring='balanced_accuracy', n_jobs=-1
    )
    return scores.mean()

lgbm_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
lgbm_study.optimize(lgbm_objective, n_trials=60, show_progress_bar=True)
print(f'Best LGBM Optuna CV: {lgbm_study.best_value:.4f}')
print(f'Best LGBM params:    {lgbm_study.best_params}')

# ── Seed search for XGBoost ───────────────────────────────────────────────────
best_seed_ba, best_seed = 0.0, 42
for seed in [0, 1, 7, 21, 42, 99]:
    m = XGBClassifier(
        **{**xgb_study.best_params,
           'scale_pos_weight': scale_pos,
           'eval_metric': 'logloss',
           'n_jobs': -1,
           'random_state': seed}
    )
    skf_s = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    s = cross_val_score(m, X_arr, y_train,
                        cv=skf_s, scoring='balanced_accuracy', n_jobs=-1)
    print(f'  XGB seed={seed}: {s.mean():.4f}')
    if s.mean() > best_seed_ba:
        best_seed_ba, best_seed = s.mean(), seed

print(f'\nBest XGB seed: {best_seed} → CV {best_seed_ba:.4f}')

# ── Final params ──────────────────────────────────────────────────────────────
hgbm_params = {
    'learning_rate': 0.016782184919286597,
    'max_iter': 1117,
    'max_leaf_nodes': 26,
    'max_depth': 5,
    'min_samples_leaf': 72,
    'l2_regularization': 2.2767559805786015,
}

xgb_params = {
    **xgb_study.best_params,
    'scale_pos_weight': scale_pos,
    'eval_metric':      'logloss',
    'random_state':     best_seed,
    'n_jobs':           -1,
}

lgbm_params = {
    **lgbm_study.best_params,
    'class_weight':  'balanced',
    'random_state':  42,
    'n_jobs':        -1,
    'verbose':       -1,
}

# ── OOF training ──────────────────────────────────────────────────────────────
oof_hgbm  = np.zeros(len(y_train))
oof_xgb   = np.zeros(len(y_train))
oof_lgbm  = np.zeros(len(y_train))
test_hgbm = np.zeros(len(X_test_te))
test_xgb  = np.zeros(len(X_test_te))
test_lgbm = np.zeros(len(X_test_te))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_arr, y_train)):
    X_tr, X_val = X_arr[tr_idx], X_arr[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    m_hgbm = HistGradientBoostingClassifier(
        class_weight='balanced', random_state=42,
        early_stopping=False, **hgbm_params
    )
    m_hgbm.fit(X_tr, y_tr)
    oof_hgbm[val_idx]  = m_hgbm.predict_proba(X_val)[:, 1]
    test_hgbm         += m_hgbm.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    m_xgb = XGBClassifier(**xgb_params)
    m_xgb.fit(X_tr, y_tr)
    oof_xgb[val_idx]   = m_xgb.predict_proba(X_val)[:, 1]
    test_xgb          += m_xgb.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    m_lgbm = LGBMClassifier(**lgbm_params)
    m_lgbm.fit(X_tr, y_tr)
    oof_lgbm[val_idx]  = m_lgbm.predict_proba(X_val)[:, 1]
    test_lgbm         += m_lgbm.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    print(f'Fold {fold+1}/{N_SPLITS} done')

print('\nOOF Balanced Accuracy per model:')
for name, oof in [('HGBM', oof_hgbm), ('XGB', oof_xgb), ('LGBM', oof_lgbm)]:
    best_ba, best_t = 0.0, 0.5
    for t in np.arange(0.1, 0.9, 0.005):
        ba = balanced_accuracy_score(y_train, (oof >= t).astype(int))
        if ba > best_ba:
            best_ba, best_t = ba, t
    print(f'  {name}: {best_ba:.4f}  (best t={best_t:.3f})')

Best trial: 55. Best value: 0.868771: 100%|██████████| 60/60 [02:05<00:00,  2.08s/it]


Best XGB Optuna CV: 0.8688
Best XGB params:    {'n_estimators': 899, 'learning_rate': 0.044925311663262746, 'max_depth': 4, 'min_child_weight': 49, 'subsample': 0.9066264844754309, 'colsample_bytree': 0.9404726184518012, 'reg_alpha': 0.5796743517191622, 'reg_lambda': 2.9719182594187536, 'gamma': 1.0300044484691284}


Best trial: 46. Best value: 0.8666: 100%|██████████| 60/60 [13:06<00:00, 13.11s/it]  


Best LGBM Optuna CV: 0.8666
Best LGBM params:    {'n_estimators': 654, 'learning_rate': 0.025695396965748477, 'max_depth': 7, 'num_leaves': 24, 'min_child_samples': 23, 'subsample': 0.8156760979769445, 'colsample_bytree': 0.7770742352579232, 'reg_alpha': 2.8693357936064383, 'reg_lambda': 4.335019813134123}
  XGB seed=0: 0.8669
  XGB seed=1: 0.8660
  XGB seed=7: 0.8674
  XGB seed=21: 0.8664
  XGB seed=42: 0.8688
  XGB seed=99: 0.8646

Best XGB seed: 42 → CV 0.8688
Fold 1/10 done
Fold 2/10 done
Fold 3/10 done
Fold 4/10 done
Fold 5/10 done
Fold 6/10 done
Fold 7/10 done
Fold 8/10 done
Fold 9/10 done
Fold 10/10 done

OOF Balanced Accuracy per model:
  HGBM: 0.8736  (best t=0.415)
  XGB: 0.8709  (best t=0.440)
  LGBM: 0.8722  (best t=0.405)


In [7]:
# ── Free blend search ─────────────────────────────────────────────────────────
best_ba, best_weights, best_threshold = 0.0, (1, 1, 1), 0.5

for w_h in [1, 2, 3]:
    for w_x in [1, 2, 3]:
        for w_l in [1, 2, 3]:
            total   = w_h + w_x + w_l
            blended = (w_h*oof_hgbm + w_x*oof_xgb + w_l*oof_lgbm) / total
            for t in np.arange(0.1, 0.9, 0.005):
                preds = (blended >= t).astype(int)
                n_pos = preds.sum()
                ba    = balanced_accuracy_score(y_train, preds)
                if ba > best_ba:
                    best_ba        = ba
                    best_weights   = (w_h, w_x, w_l)
                    best_threshold = t

print(f'Free search — OOF BA: {best_ba:.4f}  weights: {best_weights}  threshold: {best_threshold:.3f}')

# ── Constrained blend search (pred dist 1 between 4600–4700) ─────────────────
best_ba_c, best_weights_c, best_threshold_c = 0.0, (1, 1, 1), 0.5

for w_h in [1, 2, 3]:
    for w_x in [1, 2, 3]:
        for w_l in [1, 2, 3]:
            total   = w_h + w_x + w_l
            blended = (w_h*oof_hgbm + w_x*oof_xgb + w_l*oof_lgbm) / total
            for t in np.arange(0.1, 0.9, 0.005):
                preds = (blended >= t).astype(int)
                n_pos = preds.sum()
                if 4600 <= n_pos <= 4700:          # constrain to best LB range
                    ba = balanced_accuracy_score(y_train, preds)
                    if ba > best_ba_c:
                        best_ba_c        = ba
                        best_weights_c   = (w_h, w_x, w_l)
                        best_threshold_c = t

print(f'Constrained search — OOF BA: {best_ba_c:.4f}  weights: {best_weights_c}  threshold: {best_threshold_c:.3f}')

# ── Pick the better one ───────────────────────────────────────────────────────
if best_ba_c >= best_ba - 0.001:   # prefer constrained if within 0.001
    best_weights   = best_weights_c
    best_threshold = best_threshold_c
    best_ba        = best_ba_c
    print('\nUsing CONSTRAINED result')
else:
    print('\nUsing FREE result (constrained was too costly)')

print(f'\nFinal — OOF BA: {best_ba:.4f}  weights: {best_weights}  threshold: {best_threshold:.3f}')

Free search — OOF BA: 0.8738  weights: (2, 3, 3)  threshold: 0.445
Constrained search — OOF BA: 0.8310  weights: (1, 3, 3)  threshold: 0.705

Using FREE result (constrained was too costly)

Final — OOF BA: 0.8738  weights: (2, 3, 3)  threshold: 0.445


In [12]:
# Test: old hand-picked XGB + new Optuna LGBM
xgb_params_old = {
    'n_estimators': 1000,
    'learning_rate': 0.02,
    'max_depth': 5,
    'min_child_weight': 10,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_lambda': 2.0,
    'scale_pos_weight': scale_pos,
    'eval_metric': 'logloss',
    'random_state': 42,
    'n_jobs': -1,
}

oof_xgb_old  = np.zeros(len(y_train))
test_xgb_old = np.zeros(len(X_test_te))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_arr, y_train)):
    X_tr, X_val = X_arr[tr_idx], X_arr[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    m = XGBClassifier(**xgb_params_old)
    m.fit(X_tr, y_tr)
    oof_xgb_old[val_idx]  = m.predict_proba(X_val)[:, 1]
    test_xgb_old         += m.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    print(f'Fold {fold+1}/10 done')

# Check OOF BA
best_ba_old, best_t_old = 0.0, 0.5
for t in np.arange(0.1, 0.9, 0.005):
    ba = balanced_accuracy_score(y_train, (oof_xgb_old >= t).astype(int))
    if ba > best_ba_old:
        best_ba_old, best_t_old = ba, t
print(f'\nOld XGB OOF BA: {best_ba_old:.4f}  (best t={best_t_old:.3f})')

# Blend old XGB + Optuna LGBM + HGBM with constrained threshold
best_ba_mix, best_w_mix, best_t_mix = 0.0, (1,3,1), 0.5

for w_h in [1, 2, 3]:
    for w_x in [1, 2, 3]:
        for w_l in [1, 2, 3]:
            total   = w_h + w_x + w_l
            blended = (w_h*oof_hgbm + w_x*oof_xgb_old + w_l*oof_lgbm) / total
            for t in np.arange(0.1, 0.9, 0.005):
                preds = (blended >= t).astype(int)
                n_pos = preds.sum()
                if 4600 <= n_pos <= 4750:
                    ba = balanced_accuracy_score(y_train, preds)
                    if ba > best_ba_mix:
                        best_ba_mix = ba
                        best_w_mix  = (w_h, w_x, w_l)
                        best_t_mix  = t

print(f'\nMix constrained — OOF BA: {best_ba_mix:.4f}  weights: {best_w_mix}  threshold: {best_t_mix:.3f}')

# Preview pred dist on test
w_h, w_x, w_l = best_w_mix
total = w_h + w_x + w_l
test_blend_mix = (w_h*test_hgbm + w_x*test_xgb_old + w_l*test_lgbm) / total
preds_mix = (test_blend_mix >= best_t_mix).astype(int)
print(f'Pred dist — 0: {(preds_mix==0).sum()}, 1: {(preds_mix==1).sum()}')

Fold 1/10 done
Fold 2/10 done
Fold 3/10 done
Fold 4/10 done
Fold 5/10 done
Fold 6/10 done
Fold 7/10 done
Fold 8/10 done
Fold 9/10 done
Fold 10/10 done

Old XGB OOF BA: 0.8727  (best t=0.420)

Mix constrained — OOF BA: 0.8309  weights: (2, 2, 3)  threshold: 0.700
Pred dist — 0: 16763, 1: 3130


In [13]:
# Find what pred dist 1 range actually maximizes OOF BA with current models
w_h, w_x, w_l = 2, 3, 3   # best weights from free search
total   = w_h + w_x + w_l
blended = (w_h*oof_hgbm + w_x*oof_xgb + w_l*oof_lgbm) / total

print('Threshold | Pred dist 1 | OOF BA')
print('-' * 40)
for t in np.arange(0.35, 0.65, 0.005):
    preds = (blended >= t).astype(int)
    n_pos = preds.sum()
    ba    = balanced_accuracy_score(y_train, preds)
    print(f'  {t:.3f}   |    {n_pos}      | {ba:.4f}')

Threshold | Pred dist 1 | OOF BA
----------------------------------------
  0.350   |    8297      | 0.8696
  0.355   |    8250      | 0.8701
  0.360   |    8196      | 0.8707
  0.365   |    8124      | 0.8712
  0.370   |    8072      | 0.8719
  0.375   |    8016      | 0.8720
  0.380   |    7965      | 0.8718
  0.385   |    7905      | 0.8717
  0.390   |    7844      | 0.8717
  0.395   |    7790      | 0.8721
  0.400   |    7732      | 0.8717
  0.405   |    7669      | 0.8721
  0.410   |    7618      | 0.8721
  0.415   |    7571      | 0.8721
  0.420   |    7521      | 0.8721
  0.425   |    7453      | 0.8720
  0.430   |    7400      | 0.8721
  0.435   |    7355      | 0.8727
  0.440   |    7307      | 0.8729
  0.445   |    7246      | 0.8738
  0.450   |    7179      | 0.8726
  0.455   |    7133      | 0.8723
  0.460   |    7080      | 0.8725
  0.465   |    7025      | 0.8715
  0.470   |    6971      | 0.8712
  0.475   |    6918      | 0.8712
  0.480   |    6876      | 0.8710
  0.485 

In [11]:
# Quick test before committing — run this before Cell 8
w_h, w_x, w_l = 1, 3, 1
total = w_h + w_x + w_l
blended = (w_h*oof_hgbm + w_x*oof_xgb + w_l*oof_lgbm) / total

best_ba_test, best_t_test = 0.0, 0.5
for t in np.arange(0.1, 0.9, 0.005):
    preds = (blended >= t).astype(int)
    n_pos = preds.sum()
    ba    = balanced_accuracy_score(y_train, preds)
    if 4600 <= n_pos <= 4700 and ba > best_ba_test:
        best_ba_test, best_t_test = ba, t

print(f'Forced (1,3,1) constrained — OOF BA: {best_ba_test:.4f}  threshold: {best_t_test:.3f}')
blended_test = (1*test_hgbm + 3*test_xgb + 1*test_lgbm) / 5
preds_test   = (blended_test >= best_t_test).astype(int)
print(f'Pred dist — 0: {(preds_test==0).sum()}, 1: {(preds_test==1).sum()}')

Forced (1,3,1) constrained — OOF BA: 0.8290  threshold: 0.705
Pred dist — 0: 16782, 1: 3111


In [8]:
w_h, w_x, w_l = best_weights
total = w_h + w_x + w_l

test_blend = (w_h*test_hgbm + w_x*test_xgb + w_l*test_lgbm) / total
test_preds = (test_blend >= best_threshold).astype(int)

print(f'Prediction distribution — 0: {(test_preds==0).sum()}, 1: {(test_preds==1).sum()}')
print(f'OOF CV used for threshold: {best_ba:.4f}')

Prediction distribution — 0: 15103, 1: 4790
OOF CV used for threshold: 0.8738


In [ ]:
submission = pd.DataFrame({
    'id':           TEST_DATA.index,
    'subscription': test_preds
})
#submission.to_csv('submissionr2.csv', index=False)
print('Saved! Preview:')
print(submission.head())

Saved! Preview:
      id  subscription
0  37797             0
1  37798             0
2  37799             0
3  37800             0
4  37801             1
